# Final Project — Week 2: Advanced Analytics and Insights

Welcome to the second and final week of this Learning Pod project! ✴️

Last week you turned four raw CSV files into a curated dataset. This week, we will build on that work and practise advanced techniques while exploring movie rankings, how ratings change over time, and what free-text tags can tell us.

This notebook has two parts:

- **Part A — The toolbox.** A guided tour where you apply the three advanced topics of module 03: complex data types, window functions and UDFs. Every question here has one expected answer.
- **Part B — Putting it all together.** Three open questions where you choose the approach and then explain your findings in your own words. This is the part we will discuss in the learning pod.

Everything you need was covered in module **03 - PySpark Advanced**.

> Read the project [README](../README.md) before starting.

## 0. Setup

In [ ]:
import os
import time

import pandas as pd
import seaborn as sns

import pyspark.sql.functions as f
from pyspark.sql import SparkSession, Window
from pyspark.sql.types import ArrayType, DoubleType, IntegerType, StringType

spark = SparkSession.builder.appName("movielens-project-week2").getOrCreate()
sc = spark.sparkContext

## 1. Loading the curated dataset

This is where last week pays off: the curated dataset is stored as Parquet, so the schema comes with the files and there is nothing to parse, cast or clean.

The cell below rebuilds the curated dataset from the raw files **only if** you have not produced it in Week 1. If your `../curated/` folder is already there, it does nothing.

In [ ]:
CURATED_DIR = "../curated"
DATA_DIR = "../movielens-data"
DATASET_DIR = f"{DATA_DIR}/ml-latest-small"

if not os.path.exists(CURATED_DIR):
    print("No curated dataset found, rebuilding it from the raw files...")

    import urllib.request
    import zipfile

    if not os.path.exists(DATASET_DIR):
        os.makedirs(DATA_DIR, exist_ok=True)
        urllib.request.urlretrieve(
            "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip",
            f"{DATA_DIR}/ml-latest-small.zip",
        )
        with zipfile.ZipFile(f"{DATA_DIR}/ml-latest-small.zip", "r") as zip_ref:
            zip_ref.extractall(DATA_DIR)

    def read_csv(file_name, schema):
        return (
            spark.read.format("csv")
            .option("header", "true")
            .schema(schema)
            .load(f"{DATASET_DIR}/{file_name}")
        )

    (
        read_csv("movies.csv", "movieId INT, title STRING, genres STRING")
        .withColumn("release_year", f.regexp_extract(f.col("title"), r"\((\d{4})\)$", 1))
        .withColumn(
            "release_year",
            f.when(f.col("release_year") == "", None).otherwise(f.col("release_year").cast(IntegerType())),
        )
        .withColumn("title", f.trim(f.regexp_replace(f.col("title"), r"\(\d{4}\)$", "")))
        .join(read_csv("links.csv", "movieId INT, imdbId STRING, tmdbId INT"), on=["movieId"], how="left")
        .write.mode("overwrite")
        .parquet(f"{CURATED_DIR}/movies")
    )

    (
        read_csv("ratings.csv", "userId INT, movieId INT, rating DOUBLE, timestamp LONG")
        .withColumn("rating_date", f.to_date(f.from_unixtime(f.col("timestamp"))))
        .withColumn("rating_year", f.year(f.col("rating_date")))
        .withColumn("rating_month", f.month(f.col("rating_date")))
        .drop("timestamp")
        .write.mode("overwrite")
        .parquet(f"{CURATED_DIR}/ratings")
    )

    (
        read_csv("tags.csv", "userId INT, movieId INT, tag STRING, timestamp LONG")
        .withColumn("tag", f.lower(f.trim(f.col("tag"))))
        .withColumn("tag_date", f.to_date(f.from_unixtime(f.col("timestamp"))))
        .drop("timestamp")
        .dropna(subset=["tag"])
        .write.mode("overwrite")
        .parquet(f"{CURATED_DIR}/tags")
    )

    print("Done.")

In [ ]:
df_movies_raw = spark.read.parquet(f"{CURATED_DIR}/movies")
df_ratings = spark.read.parquet(f"{CURATED_DIR}/ratings")
df_tags = spark.read.parquet(f"{CURATED_DIR}/tags")

df_movies_raw.printSchema()
print(df_movies_raw.count(), "movies |", df_ratings.count(), "ratings |", df_tags.count(), "tags")

---

# Part A — The toolbox

## 2. Complex data types

The `genres` column is still a single string with the genres glued together by a `|`, and the tags live in a separate DataFrame with one row per tag. Both are a bad fit for the questions we want to ask.

Arrays, structs and maps let us keep all the information about a movie in a **single row**, which is often much more natural than spreading it over many rows.

1. Turn `genres` into an **array of strings** and add a `n_genres` column with the number of genres of each movie. Call the result `df_movies`.

Let's break this question down:
- `f.split(col, pattern)` splits a string into an array, but the pattern is a **regex**, so the `|` has to be escaped as `\|`
- Movies with no genre have the literal string `(no genres listed)`, which should become an **empty array** rather than an array with one meaningless element
- `f.array()` builds an empty array, but Spark cannot infer the type of an empty array, so it has to be cast to `array<string>`

How many movies have no genre at all? And what is the maximum number of genres a movie has?

In [ ]:
# Your code goes here

2. Which movies belong to the most genres? And how many movies are tagged as `Film-Noir`?

Hint: `f.array_contains()`.

In [ ]:
# Your code goes here

3. Add a `tags` column to `df_movies` containing the **array of distinct tags** of each movie.

Let's break this question down:
- `df_tags` has one row per tag, so the tags of a movie must first be aggregated into an array
- The same tag can be applied by several users, and we only want it once
- A movie with no tags should end up with an empty array, not with a `null`

Hint: `f.collect_set()` to aggregate, and `f.coalesce()` to replace the nulls left by the join.

In [ ]:
# Your code goes here

4. Join the ratings with the movies into `df`, cache it, and build `df_movie_stats` with one row per movie containing its number of ratings and its average rating.

Keep the `genres` and `tags` arrays in `df_movie_stats`, we will need them later.

In [ ]:
# Your code goes here

5. A **struct** groups several columns into a single one, like a small record. Create a `movie_profile` struct with the title, the release year, the average rating and the number of genres, and then read the title and the average rating back out of it.

Hint: `f.struct()` to build it, `.getField()` or the dot notation to access its fields.

In [ ]:
# Your code goes here

6. `explode()` does the opposite of `collect_set()`: it turns each element of an array into its own row. Use it to compute, per genre, the number of movies, the total number of ratings and the average rating.

Store the exploded DataFrame in `df_genre_ratings` and the aggregated one in `df_genre_stats`, both will be reused below.

In [ ]:
# Your code goes here

7. A **map** is a column of key-value pairs. Collapse `df_genre_stats` into a single row holding a map from genre to average rating, and then look up the rating of `Drama` and of `Horror`.

Let's break this question down:
- `f.struct("genre", "avg_rating")` builds a key-value pair
- `f.collect_list()` aggregates all those pairs into an array
- `f.map_from_entries()` converts an array of pairs into a map
- `f.element_at(map_column, key)` reads a value out of a map

Why would you want a map here instead of a DataFrame with two columns?

In [ ]:
# Your code goes here

## 3. Window functions

Every question in section 2 collapsed rows together. Window functions let us compute aggregations **without losing the individual rows**, which is what we need to rank things and to follow a user through time.

8. What are the **top 3 movies of each genre**, considering only movies with at least 50 ratings?

The ranking should have no gaps.

In [ ]:
# Your code goes here

9. Build `df_user_timeline`: the history of each user, ordered by date, with

- `rating_nr`: the position of the rating in the user history, starting at 1
- `days_since_prev`: how many days passed since the previous rating of that user
- `next_rating`: the rating the user will give next

Then show the timeline of user `304`.

Hint: `f.row_number()`, `f.lag()`, `f.lead()` and `f.datediff()`, all over the same window.

In [ ]:
# Your code goes here

10. Which users rate in short bursts, and which ones come back to the platform over months? Compute, per user, the average number of days between consecutive ratings and show the 5 most "bursty" ones.

In [ ]:
# Your code goes here

11. Is a user in a good mood? Compute, for every rating, the **moving average of the last 5 ratings** of that user, and compare it with the **cumulative average** since the user's very first rating.

Show both for user `304`.

Hint: `rowsBetween(-4, 0)` covers the current row and the 4 previous ones, while `rowsBetween(Window.unboundedPreceding, Window.currentRow)` covers everything up to the current row.

In [ ]:
# Your code goes here

12. Which genres grew and which ones lost ground in 2018 compared to 2017?

Hint: explode the genres, count the ratings per genre and year, and then use `f.lag()` over a window partitioned by genre and ordered by year.

In [ ]:
# Your code goes here

## 4. UDFs and Pandas UDFs

In question 18 of Week 1 you saw that sorting movies by average rating puts a movie with a single 5-star rating above *The Shawshank Redemption*. The standard fix is a **weighted rating**, which pulls averages computed over few votes towards the global average:

$$weighted\_rating = \frac{v}{v + m} \cdot R + \frac{m}{v + m} \cdot C$$

where $v$ is the number of ratings of the movie, $R$ its average rating, $C$ the global average rating of the whole platform, and $m$ the minimum number of ratings we consider meaningful.

We are going to implement it three times, exactly like in the UDFs notebook of module 03: with built-in functions, with a regular UDF and with a Pandas UDF.

In [ ]:
GLOBAL_AVG = df.select(f.avg("rating")).collect()[0][0]
MIN_RATINGS = 50

print("global average rating:", GLOBAL_AVG)

df_stats = df_movie_stats.select("movieId", "title", "n_ratings", "avg_rating").cache()
print(df_stats.count(), "movies")

13. Implement the weighted rating with **built-in functions** and time it.

Hint: `f.lit()` to use a Python value inside an expression.

In [ ]:
# Your code goes here

14. Now implement the same formula as a **regular UDF** and time it.

In [ ]:
# Your code goes here

15. Finally, implement it as a **Pandas UDF** and time it.

Remember that a Pandas UDF receives a `pd.Series` per argument instead of a single value, so the formula can be written once and applied to the whole chunk at once.

In [ ]:
# Your code goes here

16. Compare the three implementations.

In [ ]:
# Your code goes here

The built-in version should be by far the fastest one. The formula is a plain arithmetic expression, so Spark can optimise it and run it entirely inside the JVM, while both UDFs have to serialise every row, send it to a Python process and bring the result back.

The Pandas UDF is designed to be faster than the regular one because the data travels in Arrow batches instead of row by row. With only ~9700 rows that advantage does not show: the time is dominated by fixed overhead per task, so the two are practically tied, and either can come out slightly ahead. The gap only opens up with millions of rows.

**The rule of thumb: only write a UDF when the logic cannot be expressed with built-in functions.**

17. Here is a case where a UDF really is the right tool. The `tags` of a movie often repeat information we already have in `genres` (`comedy`, `horror`, ...). Write a UDF that receives the `genres` and `tags` arrays and returns only the tags that are **not** one of the movie's genres, compared case-insensitively.

Hint: the return type of the UDF is `ArrayType(StringType())`.

In [ ]:
# Your code goes here

18. Write a UDF that converts a release year into a decade label (`1994` becomes `"1990s"`, and a missing year becomes `"unknown"`), and use it together with a window function to find the **2 most rated movies of each decade**.

In [ ]:
# Your code goes here

---

# Part B — Putting it all together

You now have the full toolbox. In this part there are no step-by-step instructions: for each of the three questions below, **you** decide how to answer it with PySpark and then explain what you found. The solution notebook only shows one possible path.

## 5. Three investigations

Between them, the three questions need **a window function** (question 19), **a UDF** (question 20) and **an operation on a complex data type** (question 21).

A few tips to keep in mind:

- every question is followed by a short markdown cell where you explain **what the numbers mean**, not what the code does
- be careful with small samples: an average computed over 2 ratings is not a finding

### Question 19

**Do users become harsher or more generous as they rate more movies?**

Split the history of each user in two halves, the first half and the second half of their ratings, and compute the average rating of each half. The difference between the two is the drift of that user. How many users became more generous and how many became harsher?

Hint: `f.row_number()` and `f.count()` over a window give you the position of each rating and the size of each history, and `pivot()` puts the two halves side by side.

In [ ]:
# Your code goes here

**Your findings**

*What do these numbers mean? Write here, in your own words, what you found, how confident you are in it and what you would explore next.*

### Question 20

**Do blockbusters and long tail movies get systematically different ratings?**

Use a UDF to bucket the movies by their number of ratings: `blockbuster` (100 or more), `popular` (20 to 99), `niche` (5 to 19) and `long tail` (fewer than 5). Then compare the buckets: how many movies, how many ratings and what average rating does each one have? Finally, show the top 3 movies of each bucket.

Hint: a window partitioned by bucket and ordered by average rating gives you the ranking inside each bucket.

In [ ]:
# Your code goes here

**Your findings**

*What do these numbers mean? Write here, in your own words, what you found, how confident you are in it and what you would explore next.*

### Question 21

**Which tags are associated with the best and the worst rated movies?**

Explode the `tags` array and compute, per tag, the number of movies, the number of ratings and the average rating. Keep only the tags applied to at least 5 different movies and with at least 100 ratings, so that a single popular movie cannot carry a tag on its own. How far is each tag from the global average rating? Show the 10 best and the 10 worst tags.

Hint: `f.explode()` and `f.countDistinct()`.

In [ ]:
# Your code goes here

**Your findings**

*What do these numbers mean? Write here, in your own words, what you found, how confident you are in it and what you would explore next.*

---

## Wrap up

Congratulations, you reached the end of the specialization! ✴️

Over these two weeks you practised a data workflow: raw files, RDDs, schemas, data quality, joins, aggregations, window functions, complex data types and UDFs. You also practised turning analysis into clear findings to share with your peers.

Make sure your deliverables are complete:

- [ ] Part A answered from question 1 to 18
- [ ] Part B answered, from question 19 to 21
- [ ] each Part B answer followed by your findings, written in your own words